# Pra onde seu deputado manda dinheiro?
**Nikolas Ferreira (PL-MG) x Erika Hilton (PSOL-SP)** — emendas individuais 2023–2026

A cota de emenda individual é igual para todo deputado. A pergunta, então, não é *quanto*, mas **para onde, para quê e por qual modalidade** o dinheiro foi indicado.

Este notebook foi organizado para **auditoria**: cada etapa diz de onde vem o dado, o que o código faz, que decisão foi tomada e como conferir o resultado.

## Como auditar este notebook

### Fontes
| Fonte | O que traz | Onde é usada |
|---|---|---|
| [Portal da Transparência — Emendas parlamentares (arquivo único)](https://portaldatransparencia.gov.br/download-de-dados/emendas-parlamentares/UNICO) → `EmendasParlamentares.csv` | Uma linha por **emenda × classificação orçamentária (função/subfunção/programa/ação) × localidade de aplicação** — uma emenda pode ter várias linhas. Traz autor, tipo e valores empenhado/liquidado/pago/restos a pagar | Seções 2 a 5 |
| Mesmo zip → `EmendasParlamentares_PorFavorecido.csv` | Quem efetivamente **recebeu** o dinheiro (CNPJ/nome, natureza jurídica, município, `VALOR RECEBIDO`) | Seção 6 |
| Mesmo zip → `EmendasParlamentares_Convenios.csv` | Convênios firmados com o dinheiro da emenda (convenente, objeto, `VALOR CONVENIO`) | Seção 7 |
| TSE — votação por município 2022 | Votos por município | **Não usada neste notebook** (URL comentada no setup; ver Limitações) |

⚠️ **Localidade não é destino.** A coluna `LOCALIDADE DE APLICACAO DO RECURSO` do arquivo principal (no arquivo de convênios, `LOCALIDADE DO GASTO`) é a localidade da **programação orçamentária** — UF, `NACIONAL`, `MULTIPLO` — e **não o destino do dinheiro**. O destino real está em `PorFavorecido` (seção 6). Esse foi o erro que quase entrou na análise.

### Como reproduzir
1. Rodar **de cima para baixo** (*Restart Kernel & Run All*). A ordem das células foi corrigida para isso funcionar.
2. O download fica em cache em `data/`. O Portal atualiza os dados continuamente, então **os números mudam conforme a data do download**. A célula *Proveniência* (seção 1) imprime data e hash SHA-256 do arquivo usado — anote-os junto com qualquer número citado.
3. Para refazer com dados novos, apague `data/emendas.zip` e rode tudo de novo.

### Convenções deste documento
- **🔎 Checagem** — célula cuja saída serve para conferir que um passo funcionou (filtros, nomes de colunas, cobertura).
- **⚠️ Atenção** — decisão metodológica ou inconsistência que pode afetar a leitura do número.
- Arquivos gerados vão para `out/` (lista na última seção).

### Glossário rápido
- **Empenhado**: valor reservado no orçamento para aquela despesa (compromisso).
- **Liquidado**: etapa entre o empenho e o pagamento, quando o governo confirma que o serviço foi prestado ou o bem entregue.
- **Pago**: valor pago no próprio exercício.
- **Restos a pagar pagos (RAP pagos)**: pagamentos feitos em anos seguintes de valores empenhados antes.
- **`pago_total`** = pago + RAP pagos. É a melhor medida de "dinheiro que saiu".
- **Transferência especial ("emenda Pix")**: repasse direto a estado/município sem finalidade definida na emenda. **Transferência com finalidade definida**: repasse vinculado a um programa/ação.
- **Função / subfunção / ação**: classificação orçamentária da despesa (área → sub-área → programa específico).
- **Valor recebido** (`PorFavorecido`): valor efetivamente pago a cada favorecido.
- **Valor do convênio** (`Convenios`): ⚠️ valor **acordado** no convênio, não o pago. Serve para dizer "para quê"; **não deve ser somado** ao `pago_total`.
- **Canal**: ⚠️ agrupamento **feito nesta análise** a partir da natureza jurídica do favorecido (regras na seção 6.2), não uma categoria oficial. "Banco (intermediário)" reúne sociedades de economia mista — nestes dados, só o Banco do Brasil —, que recebem em nome de outro ente e por isso **não são o destino final**. A Caixa é empresa pública e, pela regra atual, cai em "Empresas".

### Mapa
0. Setup · 1. Extração e proveniência · 2. Filtro dos deputados e variáveis · 3. Quanto e por qual modalidade (+ comparação com todos os parlamentares) · 4. Localidade da programação (não é destino) · 5. Áreas (função/subfunção/ação) · 6. Quem recebeu (favorecidos) · 7. Convênios · 8. Limitações e checklist

## 0. Setup
Instala/importa bibliotecas e define os **parâmetros da análise**. Para replicar com outros parlamentares ou anos, altere só `DEPUTADOS` e `ANOS`.

Versões da execução original: Python 3.14.6, pandas 3.0.3, matplotlib 3.11.2, unidecode 1.4.0.

In [ ]:
# Dependências (a saída abaixo é só o log do pip)
%pip install pandas matplotlib unidecode requests

import re, zipfile
from pathlib import Path

import pandas as pd
import requests
import matplotlib.pyplot as plt  # importado para gráficos; nenhum gráfico é gerado nesta versão
from unidecode import unidecode

# Exibição: separador de milhar "," e 2 casas (formato EN). Ex.: 1,234.56 = R$ 1.234,56
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
# data/ = arquivos brutos baixados (cache) | out/ = tabelas exportadas
DATA, OUT = Path("data"), Path("out")
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)

# --- Parâmetros da análise ---------------------------------------------------
# chave = trecho do nome do autor como aparece no Portal, já normalizado por norm()
#         (maiúsculas, sem acento). uf = estado pelo qual foi eleito (usado em "fora do estado").
DEPUTADOS = {
    "NIKOLAS FERREIRA": {"uf": "MG", "cor": "#1f4e9c"},
    "ERIKA HILTON":     {"uf": "SP", "cor": "#c0392b"},
}
ANOS = range(2023, 2027)  # 2023–2026. Filtra pela coluna "ANO DA EMENDA" (ano do orçamento), não pela data do pagamento

# URLs oficiais de download
URL_EMENDAS = "https://portaldatransparencia.gov.br/download-de-dados/emendas-parlamentares/UNICO"
# TSE (votação por município 2022): NÃO usado neste notebook; guardado para um próximo post.
# URL_TSE = "https://cdn.tse.jus.br/estatistica/sead/odsele/votacao_candidato_munzona/votacao_candidato_munzona_2022.zip"

### Funções auxiliares
Todas as transformações de texto e número passam por estas funções — vale lê-las antes do resto.

In [ ]:
def norm(s):
    """Padroniza texto para comparação: remove acentos, põe em maiúsculas e colapsa espaços.
    Usada nos nomes das colunas e dos autores (ex.: 'Nome Ação' -> 'NOME ACAO')."""
    return re.sub(r"\s+", " ", unidecode(str(s)).upper().strip())

def baixar(url, destino):
    """Baixa o arquivo só se ainda não existir em disco (cache). Para forçar novo download, apague o arquivo."""
    if not destino.exists():
        print("baixando", url)
        r = requests.get(url, timeout=900); r.raise_for_status()
        destino.write_bytes(r.content)
    return destino

def ler_zip(path, padrao):
    """Lê, de dentro do zip, o primeiro CSV cujo nome casa com `padrao` (regex).
    Formato do Portal: separador ';' e encoding latin-1. Tudo é lido como texto (dtype=str)
    para não perder zeros à esquerda em códigos; números são convertidos depois com br2float()."""
    with zipfile.ZipFile(path) as z:
        nome = next(n for n in z.namelist() if re.search(padrao, n, re.I))
        return pd.read_csv(z.open(nome), sep=";", encoding="latin-1", dtype=str)

def br2float(s):
    """Converte valor no formato brasileiro ('1.234.567,89') para float. Texto inválido vira NaN."""
    return pd.to_numeric(s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False), errors="coerce")

def col(df, *trechos):
    """Acha a coluna cujo nome contém todos os `trechos`. Protege contra pequenas mudanças de layout.
    Se nenhuma coluna casar, levanta StopIteration -> sinal de que o layout do arquivo mudou."""
    return next(c for c in df.columns if all(t in c for t in trechos))

## 1. Extração — emendas
A primeira execução baixa o arquivo único do Portal (~33 MB zipado; pode demorar). Depois fica em cache em `data/`.

O zip tem três CSVs; aqui lemos o principal (`EmendasParlamentares.csv`). Os outros dois são lidos nas seções 6 e 7.

In [ ]:
# Baixa (ou usa cache) e lê o CSV principal. O regex r"\.csv$" pega o 1º .csv da lista do zip,
# que é EmendasParlamentares.csv (ver checagem do conteúdo do zip logo abaixo).
raw = ler_zip(baixar(URL_EMENDAS, DATA / "emendas.zip"), r"\.csv$")
raw.columns = [norm(c) for c in raw.columns]  # nomes de coluna sem acento/maiúsculas
print(raw.shape)  # (linhas, colunas) do arquivo bruto: todos os parlamentares, todos os anos
list(raw.columns)

### 🔎 Checagem: conteúdo do arquivo e proveniência
Confirma quais CSVs vieram no zip e quais colunas de valor existem (incluindo restos a pagar). A segunda célula registra **qual versão** do arquivo foi analisada.

In [ ]:
# 1. o que tem dentro do zip (o 1º arquivo listado é o que foi lido acima)
with zipfile.ZipFile(DATA / "emendas.zip") as z:
    for i in z.infolist():
        print(f"{i.filename:60} {i.file_size/1e6:8.1f} MB")

# 2. colunas de valor disponíveis (confirma que existe "RESTOS A PAGAR PAGOS", usada na seção 3)
print([c for c in raw.columns if "VALOR" in c])

In [ ]:
# 🔎 Proveniência: data do arquivo em disco e hash SHA-256.
# Cite estes dois valores junto com os resultados — outro auditor com o mesmo hash deve obter os mesmos números.
import hashlib, datetime as dt
f = DATA / "emendas.zip"
print(f"{f.name}: {f.stat().st_size/1e6:.1f} MB")
print("modificado em:", dt.datetime.fromtimestamp(f.stat().st_mtime).strftime("%Y-%m-%d %H:%M"))
print("sha256:", hashlib.sha256(f.read_bytes()).hexdigest())

### Mapeamento de colunas
Confira os nomes das colunas listados acima — se o layout do Portal mudar, ajuste os trechos passados a `col()`.

In [ ]:
# Nomes reais das colunas usadas daqui em diante (localizados por trecho de nome)
c_ano, c_tipo = col(raw, "ANO"), col(raw, "TIPO DE EMENDA")
c_autor, c_local = col(raw, "NOME DO AUTOR"), col(raw, "LOCALIDADE")
c_funcao, c_pago, c_emp = col(raw, "NOME FUN"), col(raw, "VALOR PAGO"), col(raw, "VALOR EMPENHADO")

# 🔎 Checagem: tipos de emenda existentes no arquivo. O filtro da seção 2 usa "INDIVIDUAL",
# que pega as duas primeiras categorias (finalidade definida + transferências especiais/"Pix").
# Emendas de relator, bancada e comissão ficam fora porque não são atribuíveis a um único deputado.
raw[c_tipo].value_counts()

## 2. Filtro dos deputados e variáveis derivadas

**Regras de filtro** (todas precisam valer):
1. Nome do autor, normalizado, **contém** a chave de `DEPUTADOS` (ex.: `"NIKOLAS FERREIRA"`).
2. `ANO DA EMENDA` em `ANOS` (2023–2026).
3. Tipo de emenda contém `"INDIVIDUAL"`.

⚠️ O casamento é por trecho de nome: um homônimo seria capturado. A checagem abaixo lista **todas as grafias originais** que entraram — deve haver exatamente uma por deputado.

In [ ]:
em = raw.copy()
em["autor"] = em[c_autor].map(norm)

# Marca a qual deputado pertence cada linha (None = nenhum dos dois)
em["deputado"] = None
for nome in DEPUTADOS:
    em.loc[em["autor"].str.contains(nome, na=False), "deputado"] = nome

# Regras 1 e 2: é um dos deputados e o ano está no período
em = em[em["deputado"].notna() & em[c_ano].astype(int).isin(ANOS)]
# Regra 3: só emendas individuais
em = em[em[c_tipo].str.contains("INDIVIDUAL", case=False, na=False)].copy()

# 🔎 Checagem: grafias originais do nome que entraram no filtro (esperado: uma por deputado)
em.groupby("deputado")[c_autor].unique()

### Variáveis derivadas
| Coluna | Regra |
|---|---|
| `pago`, `empenhado` | valores convertidos de texto BR para número |
| `pix` | `True` se o tipo é "Transferências Especiais" |
| `local` | localidade de aplicação normalizada |
| `municipio`, `uf_mun` | extraídos de localidades no formato `"CIDADE - UF"` |
| `destino` | `"Municípios"` se tem município; `"Governo do estado"` se a localidade termina em `(UF)`; senão `"Nacional / outro"` |
| `funcao` | nome da função orçamentária em formato título |

⚠️ `local`, `municipio` e `destino` vêm de `LOCALIDADE DE APLICACAO DO RECURSO`, que é a localidade da **programação orçamentária** (UF, `NACIONAL`, `MULTIPLO`), **não o destino do dinheiro**. Apesar do nome, `destino` aqui significa "onde a despesa foi programada". O destino real está em `PorFavorecido` (seção 6).

In [ ]:
em["pago"], em["empenhado"] = br2float(em[c_pago]), br2float(em[c_emp])
em["pix"] = em[c_tipo].str.contains("ESPECIA", case=False, na=False)  # transferência especial = "emenda Pix"
em["local"] = em[c_local].map(norm)

# Localidade no formato "CIDADE - UF" -> município identificado
mun = em["local"].str.extract(r"^(.*) - ([A-Z]{2})$")
em["municipio"], em["uf_mun"] = mun[0], mun[1]

# Classificação do destino (a ordem importa: a última regra que casar prevalece)
em["destino"] = "Nacional / outro"
em.loc[em["local"].str.contains(r"\(UF\)", na=False), "destino"] = "Governo do estado"
em.loc[em["municipio"].notna(), "destino"] = "Municípios"
em["funcao"] = em[c_funcao].str.title()

# 🔎 Checagem: o que caiu em "Nacional / outro". MULTIPLO/NACIONAL = localidade da programação
# orçamentária, NÃO destino do dinheiro; o destino real só aparece em PorFavorecido (seção 6).
em.loc[em.destino == "Nacional / outro", "local"].value_counts().head(20)

### 🔎 Checagem: como a localidade aparece para cada deputado
Mostra a distribuição de localidades (em número de linhas, não em R$). Quase tudo é `(UF)`, `NACIONAL` ou `MULTIPLO` — o que limita a análise geográfica da seção 4.

In [ ]:
# (célula movida: na versão original rodava antes de `em` existir)
em.groupby("deputado")["local"].value_counts().head(30)

## 3. Quanto e por qual modalidade

- `pago_total` = **pago no exercício + restos a pagar pagos** (pagamentos em anos seguintes).
- `pct_executado` = `pago_total / empenhado` — quanto do que foi reservado já saiu.
- `pct_pix` = parcela do `pago_total` que foi por transferência especial.

Saída: `out/1_resumo.csv`.

In [ ]:
em["rap_pago"] = br2float(em[col(raw, "RESTOS A PAGAR PAGOS")])
em["pago_total"] = em["pago"] + em["rap_pago"]

resumo = em.groupby("deputado").agg(
    empenhado=("empenhado", "sum"),
    pago_total=("pago_total", "sum"),
    # soma do pago_total só nas linhas marcadas como Pix
    pago_pix=("pago_total", lambda s: s[em.loc[s.index, "pix"]].sum()),
)
resumo["pct_executado"] = 100 * resumo["pago_total"] / resumo["empenhado"]
resumo["pct_pix"] = 100 * resumo["pago_pix"] / resumo["pago_total"]
resumo.to_csv(OUT / "1_resumo.csv")
resumo

🔎 **Checagem cruzada**: a mesma informação por tipo de emenda. A linha "Transferências Especiais" deve bater com `pago_pix` acima.

In [ ]:
# pago_total por tipo de emenda x deputado (R$)
pd.crosstab(em[c_tipo], em["deputado"], values=em["pago_total"], aggfunc="sum").fillna(0)

### Por ano da emenda
⚠️ Esta tabela usa `pago` (**sem** restos a pagar), então soma menos que `pago_total`.

⚠️ **2023 não aparece** porque o arquivo não tem emendas individuais desses autores com ano 2023 (provável razão: as emendas da LOA 2023 foram apresentadas em 2022, antes da posse em fev/2023). **2026 é ano em curso** — valores parciais.

In [ ]:
# pago no exercício (sem RAP) por ano da emenda
em.pivot_table(index=c_ano, columns="deputado", values="pago", aggfunc="sum", fill_value=0)

### Comparação: uso de Pix por todos os parlamentares
Para saber se o `pct_pix` de cada um é alto ou baixo, calcula-se o mesmo indicador para **todos os autores** de emendas individuais 2023–2026 no arquivo.

⚠️ "Parlamentares" aqui inclui **senadores** (também autores de emendas individuais). O agrupamento é pelo nome original do autor.

In [ ]:
# Mesma base de filtro (individual + anos), mas sem filtrar autor
base = raw[raw[c_tipo].str.contains("INDIVIDUAL", case=False, na=False) & raw[c_ano].astype(int).isin(ANOS)].copy()
base["pago_total"] = br2float(base[c_pago]) + br2float(base[col(raw, "RESTOS A PAGAR PAGOS")])
base["pix"] = base[c_tipo].str.contains("ESPECIA", case=False, na=False)

# % do pago_total que foi via Pix, por autor
por_autor = base.groupby(c_autor).apply(lambda d: 100 * d.loc[d.pix, "pago_total"].sum() / d["pago_total"].sum())
print(f"Média: {por_autor.mean():.1f}% | Mediana: {por_autor.median():.1f}% | Autores com 0% Pix: {(por_autor == 0).mean()*100:.1f}%")
por_autor.describe()

⚠️ **Corte de R$ 20 milhões** (decisão do analista): exclui autores com pouco dinheiro pago (suplentes, mandatos curtos), que distorcem os extremos 0% e 100%. Teste outros cortes para ver se a conclusão muda.

Na versão original o percentual do Nikolas estava digitado (`29.46`); agora vem de `resumo`, para não ficar desatualizado se os dados mudarem.

In [ ]:
# só parlamentares com mandato "cheio" (tira suplentes com valores pequenos que inflam o 0% e o 100%)
CORTE = 20_000_000  # R$ de pago_total no período
tot = base.groupby(c_autor)["pago_total"].sum()
filtrado = por_autor[tot > CORTE]
print(len(filtrado), f"| com 0% Pix: {(filtrado == 0).mean()*100:.1f}%")

# percentil do Nikolas (valor vem do resumo da seção 3)
pct_nikolas = resumo.loc["NIKOLAS FERREIRA", "pct_pix"]
print(f"Nikolas ({pct_nikolas:.2f}%) usa menos Pix que {(filtrado > pct_nikolas).mean()*100:.0f}% dos parlamentares")

## 4. Localidade da programação: municípios, estado ou nacional (não é o destino final)
Usa a coluna `destino` criada na seção 2. Saídas: `out/2_destino_pct.csv`, `out/3_top_municipios.csv`, `out/5_fora_do_estado.csv`.

⚠️ Estas três tabelas usam `pago` (**sem** restos a pagar).

⚠️ **Limitação importante**: `LOCALIDADE DE APLICACAO DO RECURSO` é a localidade da programação orçamentária, **não o destino do dinheiro**. A maior parte está marcada como `NACIONAL`/`MULTIPLO` ou `(UF)` — o município final não aparece aqui. Por isso "Nacional / outro" domina, o top municípios tem só uma linha e "fora do estado" dá 0%. **O destino real é analisado na seção 6 (favorecidos).**

In [ ]:
# % do pago de cada deputado por tipo de destino (linhas somam 100)
destino = em.pivot_table(index="deputado", columns="destino", values="pago", aggfunc="sum", fill_value=0)
destino_pct = (destino.div(destino.sum(1), axis=0) * 100).round(1)
destino_pct.to_csv(OUT / "2_destino_pct.csv")
destino_pct

### Top municípios (só onde a emenda já identifica o município)

In [ ]:
# top 10 municípios por deputado, em R$ pago
top = (em[em["municipio"].notna()]
       .groupby(["deputado", "municipio", "uf_mun"])["pago"].sum().reset_index()
       .sort_values("pago", ascending=False).groupby("deputado").head(10))
top.to_csv(OUT / "3_top_municipios.csv", index=False)
top

### Fora do estado de eleição
% do pago destinado a municípios de outra UF. Só considera linhas com município identificado.

In [ ]:
# fora_uf = tem município identificado E a UF é diferente da UF de eleição do deputado
em["fora_uf"] = em["uf_mun"].notna() & (em["uf_mun"] != em["deputado"].map(lambda d: DEPUTADOS[d]["uf"]))
fora = (em.groupby("deputado")
          .apply(lambda d: 100 * d.loc[d.fora_uf, "pago"].sum() / d["pago"].sum())
          .rename("pct_fora_do_estado"))
fora.to_csv(OUT / "5_fora_do_estado.csv")
fora

## 5. Áreas (função → subfunção → ação orçamentária)
Do mais geral ao mais específico. Saída: `out/4_funcao_pct.csv`.

⚠️ A tabela por função usa `pago` (sem RAP); as de subfunção e ação usam `pago_total`. As porcentagens são sempre sobre o total do próprio deputado.

Nota de leitura: **"Encargos Especiais"** é a função onde o orçamento classifica as transferências especiais (Pix) — não é uma área temática. Para saber para que serviu esse dinheiro, ver seção 6.

In [ ]:
# % do pago de cada deputado por função (colunas somam 100)
funcao = em.pivot_table(index="funcao", columns="deputado", values="pago", aggfunc="sum", fill_value=0)
funcao_pct = (funcao / funcao.sum() * 100).round(1).sort_values(list(DEPUTADOS)[0], ascending=False)
funcao_pct.to_csv(OUT / "4_funcao_pct.csv")
funcao_pct

### Subfunção, dentro das três funções que concentram o dinheiro

In [ ]:
c_subfuncao = col(raw, "NOME SUBFUN")
em["subfuncao"] = em[c_subfuncao].str.title()

# % do pago_total TOTAL de cada deputado (não só da função) por subfunção
(em[em["funcao"].isin(["Direitos Da Cidadania", "Assistência Social", "Saúde"])]
   .pivot_table(index=["funcao", "subfuncao"], columns="deputado",
                values="pago_total", aggfunc="sum", fill_value=0)
   .div(em.groupby("deputado")["pago_total"].sum()) * 100).round(1)

### Ação orçamentária, nas subfunções que diferenciam os dois

In [ ]:
c_acao = col(raw, "NOME A")  # "NOME AÇÃO" vira "NOME ACAO" após o norm
print(c_acao)  # 🔎 confirma que pegou a coluna certa

# % do pago_total de cada deputado por ação, nas subfunções selecionadas
(em[em["subfuncao"].isin(["Direitos Individuais, Coletivos E Difusos",
                           "Servicos Socioassistenciais", "Assistência Comunitária"])]
   .groupby(["deputado", "subfuncao", c_acao])["pago_total"].sum()
   .div(em.groupby("deputado")["pago_total"].sum(), level="deputado") * 100
).round(1).sort_values(ascending=False)

## 6. Quem recebeu o dinheiro (arquivo de favorecidos)
`EmendasParlamentares_PorFavorecido.csv` registra cada recebimento (`VALOR RECEBIDO`) com o favorecido, sua natureza jurídica e município. É aqui que aparece o destino que a seção 4 não mostra.

Ligação com a base principal: pela coluna **`CODIGO DA EMENDA`**.

🔎 Conferido na preparação deste documento: todas as linhas de favorecido dos dois deputados têm código presente em `em` (emendas individuais 2024–2026), ou seja, **não entram emendas fora do escopo** nesta seção.

### 6.1 Nikolas Ferreira — detalhamento por grupo

In [ ]:
fav = ler_zip(DATA / "emendas.zip", r"PorFavorecido")
fav.columns = [norm(c) for c in fav.columns]
fav["autor"] = fav[col(fav, "NOME DO AUTOR")].map(norm)
fn = fav[fav["autor"].str.contains("NIKOLAS FERREIRA", na=False)].copy()

# 🔎 Checagem: nº de recebimentos e colunas disponíveis
print(fn.shape); print(list(fn.columns))
fn.head(3).T

**Grupos** (a última regra que casar prevalece): `Saúde` se função = Saúde → `SUAS` se a ação menciona SUAS → `Pix` se é transferência especial → senão `Outros`.

🔎 A tabela mostra o total por grupo e quanto desse valor tem favorecido **"SEM INFORMAÇÃO"** (cobertura). Os totais de `Pix` devem bater com `pago_pix` da seção 3.

In [ ]:
# Atributos da emenda (função, ação, se é Pix) trazidos da base principal pelo código
cod = col(raw, "CODIGO DA EMENDA")
info = em.drop_duplicates(cod).set_index(cod)[["funcao", c_acao, "pix"]]

fn["valor"] = br2float(fn["VALOR RECEBIDO"])
fn = fn.join(info, on="CODIGO DA EMENDA")

fn["grupo"] = "Outros"
fn.loc[fn["funcao"] == "Saúde", "grupo"] = "Saúde"
fn.loc[fn[c_acao].str.contains("SUAS", na=False), "grupo"] = "SUAS"
fn.loc[fn["pix"] == True, "grupo"] = "Pix"
fn["sem_info"] = fn["FAVORECIDO"].str.contains("SEM INFORMA", na=False)

# 1. cobertura: quanto do dinheiro tem favorecido identificado
fn.groupby("grupo").agg(total=("valor", "sum"),
                        pct_sem_info=("valor", lambda s: 100 * s[fn.loc[s.index, "sem_info"]].sum() / s.sum()))

A partir daqui, só recebimentos com favorecido identificado (`ok`).

In [ ]:
ok = fn[~fn["sem_info"]]

# 2. que tipo de ente recebeu (prefeitura, fundo, estado, entidade privada) — % de cada grupo (colunas somam 100)
(ok.pivot_table(index="NATUREZA JURIDICA", columns="grupo", values="valor", aggfunc="sum", fill_value=0)
   .pipe(lambda t: (t / t.sum() * 100).round(1)))

In [ ]:
# 3. top favorecidos do Pix e do SUAS + concentração (quanto os 10 maiores levam do total do grupo)
for g in ["Pix", "SUAS"]:
    print(f"\n=== {g} ===")
    t = (ok[ok.grupo == g].groupby(["MUNICIPIO FAVORECIDO", "UF FAVORECIDO", "FAVORECIDO"])["valor"]
           .sum().sort_values(ascending=False))
    print(f"{len(t)} favorecidos | top 10 = {100 * t.head(10).sum() / t.sum():.0f}% do valor")
    print(t.head(10).map(lambda v: f"R$ {v:,.0f}").to_string())

### 6.2 Os dois deputados — canal de entrega
Agrupa a `NATUREZA JURIDICA` do favorecido em canais. Regras (por palavra-chave, na ordem):

| Contém | Canal |
|---|---|
| "Município" / "Municipal" | Prefeituras e fundos municipais |
| "Estado" / "Estadual" | Governo estadual |
| "Associação" / "Fundação Privada" / "Cooperativa" | Organizações da sociedade civil |
| "Economia Mista" | Banco (intermediário) — nestes dados, só o Banco do Brasil |
| "Sociedade" / "Empres" | Empresas (inclui empresas públicas, como Caixa e Infraero) |
| outro | Outros / órgãos federais |

⚠️ **Canal é um agrupamento desta análise**, não uma categoria do Portal. "Banco (intermediário)" recebe em nome de outro ente: **não é o destino final**. A Caixa, que também atua como intermediária, é empresa pública e cai em "Empresas" pela regra atual.

⚠️ Aqui **não** se exclui "SEM INFORMAÇÃO" (diferente de 6.1); a parcela é pequena (< 1% no Nikolas, ver 6.1).

In [ ]:
fav_both = fav[fav["autor"].str.contains("|".join(DEPUTADOS), na=False)].copy()
fav_both["deputado"] = fav_both["autor"].map(lambda a: next(d for d in DEPUTADOS if d in a))
fav_both["valor"] = br2float(fav_both["VALOR RECEBIDO"])

def canal(n):
    """Agrupa a natureza jurídica do favorecido em canais de entrega (ver tabela acima)."""
    n = str(n)
    if "Município" in n or "Municipal" in n: return "Prefeituras e fundos municipais"
    if "Estado" in n or "Estadual" in n:     return "Governo estadual"
    if "Associação" in n or "Fundação Privada" in n or "Cooperativa" in n: return "Organizações da sociedade civil"
    if "Economia Mista" in n:                return "Banco (intermediário)"
    if "Sociedade" in n or "Empres" in n:    return "Empresas"
    return "Outros / órgãos federais"

fav_both["canal"] = fav_both["NATUREZA JURIDICA"].map(canal)
t = fav_both.pivot_table(index="canal", columns="deputado", values="valor", aggfunc="sum", fill_value=0)
# % do valor recebido de cada deputado por canal (colunas somam 100)
(t / t.sum() * 100).round(1).sort_values("ERIKA HILTON", ascending=False)

### 🔎 Checagem cruzada: base principal × `PorFavorecido`
Duas bases independentes devem dar o mesmo total: o `pago_total` (seção 3) e a soma do `VALOR RECEBIDO` dos favorecidos. Se batem, o cruzamento pelo código da emenda está correto.

Resultado na execução original (arquivo de 05/11/2026):
| Deputado | `pago_total` | Soma `PorFavorecido` | Diferença |
|---|---|---|---|
| Nikolas Ferreira | R$ 84.194.825,77 | R$ 84.190.742,62 | R$ 4.083,15 (0,005%) — duas emendas de educação pagas como restos a pagar |
| Erika Hilton | R$ 77.689.628,83 | R$ 77.389.628,83 | **R$ 300.000,00 (0,39%)** — toda na emenda `202543680008` (LGBTQIA+): R$ 5,8 mi pagos, R$ 5,5 mi com favorecido registrado |

⚠️ A diferença da Erika está **em aberto**: provavelmente um pagamento ainda sem favorecido publicado no Portal. Vale conferir de novo com um download mais recente.

In [ ]:
# Totais por deputado nas duas bases
cmp = pd.DataFrame({
    "pago_total (principal)": em.groupby("deputado")["pago_total"].sum(),
    "recebido (PorFavorecido)": fav_both.groupby("deputado")["valor"].sum(),
})
cmp["diferenca"] = cmp.iloc[:, 0] - cmp.iloc[:, 1]
cmp["dif_%"] = 100 * cmp["diferenca"] / cmp.iloc[:, 0]
print(cmp.to_string(), "\n")

# Emendas onde as duas bases não batem (diferença > R$ 1), para investigar
por_emenda = pd.concat({
    "pago_total": em.groupby(["deputado", cod])["pago_total"].sum(),
    "recebido": fav_both.groupby(["deputado", "CODIGO DA EMENDA"])["valor"].sum(),
}, axis=1).fillna(0)
por_emenda["diferenca"] = por_emenda["pago_total"] - por_emenda["recebido"]
por_emenda[por_emenda["diferenca"].abs() > 1]

## 7. Convênios — o "para quê" em texto
`EmendasParlamentares_Convenios.csv` traz o **objeto** de cada convênio (descrição do que será feito) e o convenente. Ligação pela coluna de código da emenda.

⚠️ `VALOR CONVENIO` é o valor **acordado** no convênio, não o pago. Serve para dizer "para quê"; **não some com `pago_total`**.

⚠️ Só cobre dinheiro que passou por **convênio**. Repasses fundo a fundo (ex.: saúde, SUAS) e transferências especiais não aparecem aqui — por isso os totais são bem menores que o pago.

In [ ]:
conv = ler_zip(DATA / "emendas.zip", r"Convenios")
conv.columns = [norm(c) for c in conv.columns]
print(conv.shape)
print(list(conv.columns))

# convênios ligados às emendas do Nikolas
c_cod_conv = next(c for c in conv.columns if "EMENDA" in c and "COD" in c)
cods_nikolas = set(em.loc[em["deputado"] == "NIKOLAS FERREIRA", cod])
cv = conv[conv[c_cod_conv].isin(cods_nikolas)]
print(cv.shape)  # 🔎 nº de convênios encontrados
cv.head(5).T

### 7.1 Nikolas Ferreira
⚠️ O item 3 volta **vazio**: nenhuma emenda de assistência social do Nikolas virou convênio. A pergunta sobre o Banco do Brasil (que aparece como favorecido na seção 6.2) **fica em aberto** com estes dados.

In [ ]:
cv = cv.copy()
cv["valor_conv"] = br2float(cv["VALOR CONVENIO"])

# 1. quanto em convênio por área
print(cv.groupby("NOME FUNCAO")["valor_conv"].agg(["sum", "count"]).to_string())

# 2. objetos mais frequentes (resume o "para quê")
print("\n", cv.groupby("OBJETO CONVENIO")["valor_conv"].sum().sort_values(ascending=False).head(10)
        .map(lambda v: f"R$ {v:,.0f}").to_string())

# 3. assistência social: quem é o convenente (resolve o Banco do Brasil?)
print("\n", cv[cv["NOME FUNCAO"].str.contains("Assist", na=False)]
        .groupby(["CONVENENTE", "OBJETO CONVENIO"])["valor_conv"].sum()
        .sort_values(ascending=False).head(10).map(lambda v: f"R$ {v:,.0f}").to_string())

### 7.2 Erika Hilton

In [ ]:
# convênios ligados às emendas da Erika: total por área e 15 maiores (convenente + objeto)
cods_erika = set(em.loc[em["deputado"] == "ERIKA HILTON", cod])
ce = conv[conv[c_cod_conv].isin(cods_erika)].copy()
ce["valor_conv"] = br2float(ce["VALOR CONVENIO"])
print(ce.groupby("NOME FUNCAO")["valor_conv"].agg(["sum", "count"]).to_string())
print("\n", ce.groupby(["CONVENENTE", "OBJETO CONVENIO"])["valor_conv"].sum()
        .sort_values(ascending=False).head(15).map(lambda v: f"R$ {v:,.0f}").to_string())

## 8. Limitações, pontos de atenção e checklist

### Limitações conhecidas
1. **Dados vivos.** O Portal atualiza os arquivos; números dependem da data do download (ver *Proveniência*). 2026 é parcial.
2. **Duas medidas de pago.** Seções 3 (resumo), 5 (subfunção/ação) usam `pago_total` (com restos a pagar); a tabela por ano (3), as de destino (4) e a de função (5) usam `pago` (sem RAP). Comparar números entre essas tabelas exige cuidado.
3. **Localidade não é destino.** `LOCALIDADE DE APLICACAO DO RECURSO` é a localidade da programação orçamentária (`NACIONAL`, `MULTIPLO`, `(UF)`), não o destino do dinheiro. Análises geográficas devem usar o arquivo de favorecidos (seção 6).
4. **Favorecido ≠ beneficiário final.** Fundos estaduais, bancos operadores (Banco do Brasil) e entidades podem repassar o dinheiro adiante; o arquivo mostra só o primeiro recebedor.
5. **Convênios cobrem só parte do dinheiro** (seção 7); fundo a fundo e Pix não têm objeto descrito.
6. **Benchmark de Pix** inclui senadores e depende do corte arbitrário de R$ 20 mi.
7. **TSE não usado.** `URL_TSE` está comentada no setup (guardada para um próximo post). Existem `data/tse_2022.zip` e `out/base_votos_emendas.csv` na pasta, mas **nenhuma célula deste notebook** lê o TSE ou gera esse CSV. O cruzamento votos × emendas não é reprodutível a partir daqui.
8. **Casamento por nome.** Homônimos seriam capturados; a checagem da seção 2 mostra que não houve.

### Checklist para o auditor
- [ ] Rodou *Restart & Run All* sem erros
- [ ] Anotou data e SHA-256 do `emendas.zip` (seção 1)
- [ ] Colunas esperadas existem (seção 1) e `col()` não deu `StopIteration`
- [ ] Uma grafia de nome por deputado (seção 2)
- [ ] `pago_pix` (seção 3) = linha "Transferências Especiais" do crosstab = total do grupo `Pix` (seção 6.1)
- [ ] % "sem informação" de favorecido é baixo (seção 6.1)
- [ ] Soma de `PorFavorecido` bate com `pago_total` (checagem cruzada da seção 6); diferenças listadas por emenda
- [ ] Não somou `VALOR CONVENIO` com valores pagos (seção 7)
- [ ] Testou outro corte além de R$ 20 mi no benchmark de Pix

### Arquivos gerados em `out/`
| Arquivo | Conteúdo | Medida |
|---|---|---|
| `1_resumo.csv` | empenhado, pago total, pago via Pix, % executado, % Pix | `pago_total` |
| `2_destino_pct.csv` | % por tipo de destino | `pago` |
| `3_top_municipios.csv` | top 10 municípios por deputado | `pago` |
| `4_funcao_pct.csv` | % por função orçamentária | `pago` |
| `5_fora_do_estado.csv` | % destinado a municípios de outra UF | `pago` |